# Model Building and Training

Task 2: stratified split → SMOTE on **train only** → Logistic Regression baseline → XGBoost ensemble (tuned) → Stratified 5-Fold CV → model selection.

Metrics: **AUC-PR**, **F1**, confusion matrix (not accuracy).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.data_loader import load_creditcard, load_fraud_data, load_ip_country
from src.evaluation import plot_confusion_matrix, plot_pr_curve
from src.features import engineer_fraud_features, select_model_features_creditcard, select_model_features_fraud
from src.modeling import (
    cross_validate_model,
    make_logistic_regression,
    make_xgboost,
    run_creditcard_modeling,
    run_fraud_modeling,
    save_model,
)
from src.preprocessing import clean_creditcard_data, clean_fraud_data

RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
MODELS = ROOT / "models"
MODELS.mkdir(parents=True, exist_ok=True)

## Resampling justification

**Choice: SMOTE on the training set only.**

- Fraud is a small minority; accuracy would look high while missing most fraud.
- **Undersampling** throws away legitimate transactions that help calibrate the decision boundary.
- **SMOTE** synthesizes minority examples in feature space, improving recall without discarding majority signal.
- Applied **after** the stratified split (and after encoding/scaling) so the test set stays an honest estimate of production performance.

## A. E-commerce (`Fraud_Data`)

In [ ]:
fraud_path = PROC / "fraud_features.csv"
if fraud_path.exists():
    fraud = pd.read_csv(fraud_path, parse_dates=["signup_time", "purchase_time"])
else:
    fraud = engineer_fraud_features(
        clean_fraud_data(load_fraud_data(raw_dir=RAW), load_ip_country(raw_dir=RAW))
    )

X_f, y_f = select_model_features_fraud(fraud)
print(X_f.shape, y_f.value_counts(normalize=True).to_dict())

In [ ]:
# Set tune=True for GridSearch (slower); False for a quicker first pass
fraud_results = run_fraud_modeling(X_f, y_f, resample_method="smote", tune=True)

print("=== Class distribution before / after SMOTE (train) ===")
display(fraud_results["resample_report"]["before"])
display(fraud_results["resample_report"]["after"])
print(fraud_results["resample_report"]["justification"])

print("\n=== Metrics comparison ===")
display(fraud_results["comparison"])
print("Selected model:", fraud_results["best_name"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, name in zip(axes, ["LogisticRegression", "XGBoost"]):
    cm = fraud_results["metrics"][name]["confusion_matrix"]
    plot_confusion_matrix(cm, title=f"Fraud_Data — {name}", ax=ax)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
for name, model in fraud_results["models"].items():
    prob = model.predict_proba(fraud_results["X_test"])[:, 1]
    plot_pr_curve(fraud_results["y_test"], prob, label=name, ax=ax)
plt.show()

In [ ]:
# Stratified 5-Fold CV on (preprocessed + SMOTE is train-only; CV below uses scaled features w/ class_weight / scale_pos_weight)
# For a clean CV estimate we re-fit on full X with imbalance-aware estimators:
from src.pipeline import FRAUD_CATEGORICAL, FRAUD_NUMERICAL, build_preprocessor

pre = build_preprocessor(X_f, categorical=FRAUD_CATEGORICAL, numerical=FRAUD_NUMERICAL)
X_all = pre.fit_transform(X_f)

cv_lr = cross_validate_model(make_logistic_regression(), X_all, y_f, n_splits=5)
pos = int((y_f == 1).sum()); neg = int((y_f == 0).sum())
cv_xgb = cross_validate_model(make_xgboost(scale_pos_weight=neg / max(pos, 1)), X_all, y_f, n_splits=5)

print("LR CV:"); display(cv_lr)
print("XGB CV:"); display(cv_xgb)

In [ ]:
save_model(fraud_results["best_model"], MODELS / "fraud_best_model.joblib")
save_model(fraud_results["preprocessor"], MODELS / "fraud_preprocessor.joblib")
save_model(fraud_results["feature_names"], MODELS / "fraud_feature_names.joblib")
print("Saved e-commerce artifacts to", MODELS)

## B. Bank credit card

In [ ]:
cc_path = PROC / "creditcard_clean.csv"
if cc_path.exists():
    cc = pd.read_csv(cc_path)
else:
    cc = clean_creditcard_data(load_creditcard(raw_dir=RAW))

X_c, y_c = select_model_features_creditcard(cc)
# Optional: subsample for faster iteration on huge files
# from sklearn.model_selection import train_test_split
# X_c, _, y_c, _ = train_test_split(X_c, y_c, train_size=0.3, stratify=y_c, random_state=42)

cc_results = run_creditcard_modeling(X_c, y_c, resample_method="smote", tune=False)
display(cc_results["resample_report"]["before"])
display(cc_results["resample_report"]["after"])
display(cc_results["comparison"])
print("Selected:", cc_results["best_name"])

save_model(cc_results["best_model"], MODELS / "creditcard_best_model.joblib")
save_model(cc_results["preprocessor"], MODELS / "creditcard_preprocessor.joblib")
save_model(cc_results["feature_names"], MODELS / "creditcard_feature_names.joblib")

## Model selection writeup

1. **Baseline (Logistic Regression)** — fast, coefficients are directionally interpretable, good sanity check.
2. **Ensemble (XGBoost)** — captures non-linear interactions (device velocity × time-since-signup, PCA combinations).
3. **Selection rule** — prefer the model with higher **AUC-PR** on the held-out stratified test set; break ties with **F1** and business cost of FN vs FP.
4. Typically XGBoost wins on AUC-PR for both streams while remaining SHAP-explainable via `TreeExplainer` (next notebook).

Artifacts under `models/` feed Task 3 explainability.